# Single SHP Run Test

Independent single-SHP bridge run:
- one SHP observed cases from THL
- national Finland timeline -> interventions
- one simulation run
- scaled simulated cases vs observed cases plot


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "extensions" / "scenario_api").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate extensions/scenario_api from current working directory")

extensions_dir = repo_root / "extensions"
if str(extensions_dir) not in sys.path:
    sys.path.insert(0, str(extensions_dir))

from scenario_api import (
    get_default_shp_region_config,
    population_scale_factor,
    load_observed_cases_timeseries_for_region,
    load_timeline_events_from_processed,
    default_mask_mapping_profile,
    default_contact_policy_mapping_profile,
    default_testing_tracing_mapping_profile,
    default_mask_effectiveness_profiles,
    map_timeline_events_to_interventions,
    run_single_shp_cases_scenario,
)

import pandas as pd
import matplotlib.pyplot as plt

## B) Choose one SHP

In [ ]:
shp_name = "Helsinki and Uusimaa Hospital District"
simulated_population = 200000
reference_start_date = "2020-01-01

## C) Build RegionConfig

In [ ]:
region_config = get_default_shp_region_config(
    name=shp_name,
    simulated_population=simulated_population,
)
scale_factor = population_scale_factor(region_config)

print("Region:", region_config.name)
print("Real population:", region_config.real_population)
print("Simulated population:", region_config.simulated_population)
print("Scale factor:", scale_factor)

## D) Load observed THL cases for SHP

In [ ]:
thl_processed_path = repo_root / "data" / "processed" / "thl_cases_2020_2022_processed_daily.csv"

observed_ts = load_observed_cases_timeseries_for_region(
    processed_path=thl_processed_path,
    region=shp_name,
    region_level="hospital_district",
)

print("Observed date range:", observed_ts.metadata.get("date_min"), "->", observed_ts.metadata.get("date_max"))
print("Observed first values:", observed_ts.values[:10])
print("Observed metadata:", observed_ts.metadata)

## E) Load timeline + map to interventions

In [ ]:
timeline_processed_path = repo_root / "data" / "processed" / "oxcgrt_finland_2020_2022_timeline.csv"
timeline_events = load_timeline_events_from_processed(timeline_processed_path)

# First single-SHP bridge supports this subset explicitly.
supported_event_types = {
    "facial_coverings",
    "school_closing",
    "workplace_closing",
    "gathering_restrictions",
    "stay_at_home",
    "public_events",
    "internal_movement",
    "testing_policy",
    "contact_tracing",
}
timeline_events_supported = [e for e in timeline_events if e.event_type in supported_event_types]

mask_mapping_profile = default_mask_mapping_profile()
contact_mapping_profile = default_contact_policy_mapping_profile()
testing_tracing_mapping_profile = default_testing_tracing_mapping_profile()
mask_profiles = default_mask_effectiveness_profiles()

mapped_interventions = map_timeline_events_to_interventions(
    events=timeline_events_supported,
    mask_mapping_profile=mask_mapping_profile,
    contact_mapping_profile=contact_mapping_profile,
    testing_tracing_mapping_profile=testing_tracing_mapping_profile,
    mask_profiles=mask_profiles,
    reference_start_date=reference_start_date,
)

print("Timeline events (all):", len(timeline_events))
print("Timeline events (supported subset):", len(timeline_events_supported))
print("Mapped interventions:", len(mapped_interventions))

pd.DataFrame([
    {
        "type": type(i).__name__,
        "name": i.name,
        "start": i.start,
        "end": i.end,
        "details": getattr(i, "multipliers", None) or getattr(i, "network_mix", None) or getattr(i, "config", None),
    }
    for i in mapped_interventions[:12]
])

## F) Run one SHP simulation

In [ ]:
sim_steps = len(observed_ts.times)

result, scaled_simulated_ts = run_single_shp_cases_scenario(
    region_config=region_config,
    timeline_events=timeline_events,
    mask_mapping_profile=mask_mapping_profile,
    contact_mapping_profile=contact_mapping_profile,
    testing_tracing_mapping_profile=testing_tracing_mapping_profile,
    mask_profiles=mask_profiles,
    reference_start_date=reference_start_date,
    simulation_steps=sim_steps,
    use_openabm=False,
)

print("Simulation steps:", sim_steps)
print("OpenABM used:", result.metadata.get("openabm_used"))
print("Raw outputs keys:", list(result.raw_outputs.keys()))

## G) Inspect raw vs scaled simulated cases

In [ ]:
raw_cases = result.raw_outputs["cases"]
scaled_cases = scaled_simulated_ts.values

print("Scale factor used:", scaled_simulated_ts.metadata.get("scale_factor"))
print("Raw simulated first values:", raw_cases[:10])
print("Scaled simulated first values:", scaled_cases[:10])

## H) Plot observed vs scaled simulated cases

In [ ]:
obs_dates = pd.to_datetime(observed_ts.times)
obs_values = pd.Series(observed_ts.values, dtype=float)

ref_ts = pd.Timestamp(reference_start_date)
sim_dates = [ref_ts + pd.Timedelta(days=int(t)) for t in scaled_simulated_ts.times]
sim_values = pd.Series(scaled_simulated_ts.values, dtype=float)

n = min(len(obs_dates), len(sim_dates), len(obs_values), len(sim_values))
obs_dates = obs_dates[:n]
obs_values = obs_values.iloc[:n]
sim_dates = sim_dates[:n]
sim_values = sim_values.iloc[:n]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(obs_dates.to_numpy(), obs_values.to_numpy(), label=f"Observed THL cases ({shp_name})", linewidth=2)
ax.plot(pd.Series(sim_dates).to_numpy(), sim_values.to_numpy(), label="Scaled simulated cases", linewidth=2)
ax.set_title(f"Single SHP run comparison: {shp_name}")
ax.set_xlabel("Date")
ax.set_ylabel("Cases")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## I) Short interpretation

In [ ]:
peak_obs = float(obs_values.max()) if len(obs_values) else float("nan")
peak_sim = float(sim_values.max()) if len(sim_values) else float("nan")

if peak_sim > 2.5 * peak_obs or peak_obs > 2.5 * peak_sim:
    magnitude_note = "magnitude is wildly off"
else:
    magnitude_note = "magnitude is at least in the same rough order"

print("Pipeline worked end-to-end: yes")
print("Timing rough plausibility: visually inspect peaks/turning points in plot")
print("Magnitude assessment:", magnitude_note)
print("No calibration performed in this notebook.")